<a href="https://colab.research.google.com/github/bigwisu/citrus/blob/main/citrus.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# CITRUS: Cluster-based Interactive Truncation for Retrieval Using Semantics

**Author:** Wisu Suntoyo

## Overview
This repository contains the replication code for the paper *"From Static Keywords to Adaptive Vectors: The CITRUS Framework"*. It implements a Human-in-the-Loop vector retrieval workflow using **IBM Granite** embeddings.

## How to Run
1. **Open in Colab:** Click the badge above to open `citrus.ipynb`.
2. **Set Credentials:**
   - This notebook requires access to **IBM watsonx.ai**.
   - In Colab, go to the **Secrets Manager** (Key icon on the left).
   - Add `WATSONX_API_KEY`, `WATSONX_PROJECT_ID` and `WATSONX_URL`
3. **Data Access:**
   - Raw Scopus data cannot be redistributed due to copyright.
   - However, the **final screened list** is available in `citrus_final_screening_list.csv`.
   - To replicate from scratch, place your own Scopus CSV export in your Google Drive folder defined in the ingestion cell.

## Dependencies
- `ibm-watsonx-ai`
- `sqlite-vec`
- `kneed` (For knee/elbow detection)
- `seaborn` / `matplotlib`

# Phase I: Ingestion & Control Group Establishment

**Objective:**  
To merge two disjoint bibliometric corpora (Technological vs. Managerial) while mathematically quantifying the pre-existing intersection between them.

**Methodological Logic:**
1.  **Independent Analysis:** We first analyze the *GenAI Corpus* and *PDM Corpus* separately to extract their unique identifiers.
2.  **Strict Intersection (The Control Group):** Before merging, we calculate the set intersection ($A \cap B$).
    *   *Significance:* These papers represent the existing "bridge" between GenAI and PDM found via keywords. They serve as the **Boolean Control Group** ($n \approx 40$) to validate the recall gain of the vector search.
3.  **Tagging:** We create a binary flag `is_boolean_control` to track these baseline papers throughout the vector pipeline.

In [ ]:
# @title List CSVs in Google Drive

from google.colab import drive
import os

drive.mount('/content/drive')

file_path = '/content/drive/MyDrive/citrus-csv'

# List all files and directories in the specified path
all_files = os.listdir(file_path)

# Print the list of files
csv_files = [file for file in all_files if file.endswith('.csv')]
print(csv_files)

## Ingestion & Control Group Establishment

To merge two disjoint bibliometric corpora (Technological vs. Managerial) while mathematically identifying the pre-existing intersection between them.

In [ ]:
import pandas as pd
import os

# 1. Define File Sources (Dynamic Matching)
all_files_in_dir = [f for f in os.listdir(file_path) if f.endswith('.csv') and 'scopus' in f.lower()]

genai_files = [f for f in all_files_in_dir if "GenAI" in f]
pdm_files = [f for f in all_files_in_dir if "PDM" in f]

print(f"Loading GenAI Sources: {genai_files}")
print(f"Loading PDM Sources: {pdm_files}")

# 2. Build Independent Sets for Control Group Logic
def extract_identifiers(file_list):
    ids = set()
    titles = set()
    raw_df_list = []

    for file in file_list:
        try:
            # Construct the full file path
            full_path = os.path.join(file_path, file)
            df = pd.read_csv(full_path)
            # Create normalized title for robust matching
            df['norm_title'] = df['Title'].astype(str).str.lower().str.strip()

            ids.update(df['DOI'].dropna().unique())
            titles.update(df['norm_title'].unique())
            raw_df_list.append(df)
        except Exception as e:
            print(f"Error reading {file}: {e}")

    return ids, titles, raw_df_list

print("...Analyzing GenAI Corpus...")
genai_ids, genai_titles, genai_dfs = extract_identifiers(genai_files)

print("...Analyzing PDM Corpus...")
pdm_ids, pdm_titles, pdm_dfs = extract_identifiers(pdm_files)

# 3. Calculate the Strict Intersection (The Control Group)
control_dois = genai_ids.intersection(pdm_ids)
control_titles = genai_titles.intersection(pdm_titles)

print(f"\n[AUDIT] Strict Cross-Domain Overlap: {len(control_dois)} DOIs and {len(control_titles)} Titles.")

# 4. Merge Everything
all_dfs = genai_dfs + pdm_dfs
combined_df = pd.concat(all_dfs, ignore_index=True)
initial_count = len(combined_df)

# 5. Tag the Control Group in the Main DataFrame
combined_df['is_boolean_control'] = False
mask_doi = combined_df['DOI'].isin(control_dois)
mask_title = combined_df['norm_title'].isin(control_titles)
combined_df.loc[mask_doi | mask_title, 'is_boolean_control'] = True

# 6. Deduplicate (The Cleaning)
# 6.1 Sort to ensure if a duplicate exists, we prefer the one tagged True (though both should be True now)
combined_df = combined_df.sort_values(by='is_boolean_control', ascending=False)

# 6.2 Drop duplicates by DOI
combined_df = combined_df.drop_duplicates(subset=['DOI'], keep='first')

# 6.3 Drop duplicates by Title
combined_df = combined_df.drop_duplicates(subset=['norm_title'], keep='first')

final_count = len(combined_df)
control_count_final = len(combined_df[combined_df['is_boolean_control'] == True])

print("=" * 40)
print(f"Total Raw Records Processed: {initial_count}")
print(f"Total Duplicates Removed:    {initial_count - final_count}")
print(f"FINAL UNIQUE CORPUS:         {final_count}")
print(f"CONTROL GROUP (Overlap):     {control_count_final}")
print("=" * 40)

# Check
if control_count_final == 0 and len(control_dois) > 0:
    print("WARNING: Control group lost during deduplication. Check logic.")
else:
    print("SUCCESS: Data is clean and tagged.")

In [21]:
def create_embedding_field(df: pd.DataFrame) -> pd.DataFrame:
    # 1. create a working copy to avoid SettingWithCopy warnings on the original view
    work_df = df.copy()

    # 2. Pre-processing: Fill NaNs for string construction
    # We don't want "nan" appearing in our text representation
    fill_cols = ['Authors', 'Title', 'Source title', 'Volume', 'Issue',
                 'Page start', 'Page end', 'DOI', 'Abstract']
    for col in fill_cols:
        if col in work_df.columns:
            work_df[col] = work_df[col].fillna('').astype(str) # Explicitly cast to string after filling NaN

    # Ensure numerical columns are clean
    work_df['Year'] = work_df['Year'].fillna(0).astype(int)
    work_df['Cited by'] = work_df['Cited by'].fillna(0).astype(int)

    # 3. Construct the final Embedding Text
    # We use prefixes to help the model distinguish sections.
    # Structure:
    # [Title]
    # [Content: Abstract]

    print("Constructing final embedding text...")
    work_df['embedding_text'] = (
        "Title: " + work_df['Title'] + "\n" +
        "Abstract:\n" + work_df['Abstract']
    )

    return work_df

In [ ]:
batch = combined_df.copy()
# Usage
df_processed = create_embedding_field(batch)

## Phase II: High-Dimensional Projection

**Model Architecture:** IBM Granite (`ibm/slate-125m-english-rtrvr-v2`)  
**Dimensionality:** $\mathbb{R}^{768}$

In this phase, we project the textual metadata (Title + Abstract) of the unique corpus ($N=52,108$) into a semantic vector space. Unlike keyword matching, this dense vector representation captures the *contextual intent* of the research, allowing us to identify papers that discuss "Silicon Sampling" concepts without necessarily using that exact keyword.

*Note: Data is processed in batches to adhere to API rate limits and ensure memory stability.*

In [12]:
!pip install -U ibm-watsonx-ai -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.8/58.8 kB 5.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 48.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 139.6/139.6 kB 12.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 52.6 MB/s eta 0:00:00


In [18]:
from ibm_watsonx_ai import Credentials
from ibm_watsonx_ai.foundation_models import Embeddings
from ibm_watsonx_ai.metanames import EmbedTextParamsMetaNames as EmbedParams
from ibm_watsonx_ai.foundation_models.utils.enums import EmbeddingTypes
from google.colab import userdata

WATSONX_URL = userdata.get('WATSONX_URL').split('\r\n')[0].strip()
WATSONX_API_KEY = userdata.get('WATSONX_API_KEY').split('\r\n')[0].strip()
WATSONX_PROJECT_ID = userdata.get('WATSONX_PROJECT_ID').split('\r\n')[0].strip()

def get_watsonx_embedding(text):
    credentials = Credentials(
        url=WATSONX_URL,
        api_key=WATSONX_API_KEY,
    )

    embed_params = {
        EmbedParams.TRUNCATE_INPUT_TOKENS: 512,
        EmbedParams.RETURN_OPTIONS: {
            'input_text': True
        }
    }

    embedding = Embeddings(
        model_id="ibm/slate-125m-english-rtrvr-v2",
        params=embed_params,
        credentials=credentials,
        project_id=WATSONX_PROJECT_ID
    )

    embedding_vector = embedding.embed_query(text=text)

    return embedding_vector

In [ ]:
import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from ibm_watsonx_ai import Credentials
from ibm_watsonx_ai.foundation_models import Embeddings
from ibm_watsonx_ai.metanames import EmbedTextParamsMetaNames as EmbedParams
from google.colab import userdata

def embed_dataframe_column(df: pd.DataFrame, batch_size: int = 100) -> pd.DataFrame:
    """Generates embeddings for the 'embedding_text' column of a DataFrame in batches using Watsonx.ai.

    Args:
        df (pd.DataFrame): The input DataFrame containing an 'embedding_text' column.
        batch_size (int): The number of records to process in each batch for embedding generation.

    Returns:
        pd.DataFrame: The DataFrame with a new 'embedding' column containing the generated embeddings.
    """
    all_embeddings = []
    num_batches = (len(df) + batch_size - 1) // batch_size

    print(f"Generating embeddings for {len(df)} records in {num_batches} batches using Watsonx.ai...")

    # Retrieve Watsonx.ai credentials and project ID globally defined
    WATSONX_URL = userdata.get('WATSONX_URL').split('\r\n')[0].strip()
    WATSONX_API_KEY = userdata.get('WATSONX_API_KEY').split('\r\n')[0].strip()
    WATSONX_PROJECT_ID = userdata.get('WATSONX_PROJECT_ID').split('\r\n')[0].strip()

    credentials = Credentials(
        url=WATSONX_URL,
        api_key=WATSONX_API_KEY,
    )

    embed_params = {
        EmbedParams.TRUNCATE_INPUT_TOKENS: 512
    }

    # Instantiate the Embeddings model once
    embedding_model = Embeddings(
        model_id="ibm/slate-125m-english-rtrvr-v2", # Fixed model ID for Watsonx.ai
        params=embed_params,
        credentials=credentials,
        project_id=WATSONX_PROJECT_ID
    )

    for i in tqdm(range(0, len(df), batch_size), desc="Embedding batches"):
        batch_texts = df['embedding_text'].iloc[i : i + batch_size].tolist()
        cleaned_batch_texts = [text if pd.notna(text) and text.strip() != '' else ' ' for text in batch_texts]

        try:
            # Use embed_documents for batch processing
            batch_vectors = embedding_model.embed_documents(texts=cleaned_batch_texts)
            all_embeddings.extend(batch_vectors)
        except Exception as e:
            print(f"Error generating embeddings for batch {i // batch_size + 1}: {e}")
            all_embeddings.extend([None] * len(cleaned_batch_texts)) # Fill with None for failed entries

    df['embedding'] = all_embeddings
    print(f"Finished generating embeddings. Added 'embedding' column to DataFrame with {len(all_embeddings)} embeddings.")
    return df

print("Embedding function `embed_dataframe_column` redefined to use Watsonx.ai.")

In [ ]:
df_with_embeddings = embed_dataframe_column(df_processed.copy())
print("DataFrame with embeddings generated. Displaying the first few rows with the new 'embedding' column.")
display(df_with_embeddings.head())

# [Optional] Backup/Recover Embeddings

Due to state of Colab Notebook, it is advisable to save Embeddings Data Frame and Recover if Colab instance reconnects

## Save downloaded embeddings DF as Parquet in Google Drive

In [ ]:
import os

# Ensure pyarrow is installed for Parquet support
# This might already be installed in Colab, but good to ensure
try:
    import pyarrow
except ImportError:
    !pip install pyarrow

output_filename = 'df_processed_with_embeddings.parquet'
output_filepath = os.path.join(file_path, output_filename)

# Save the DataFrame to Parquet
df_with_embeddings.to_parquet(output_filepath, index=False)

print(f"'df_with_embeddings' DataFrame saved to '{output_filepath}' in Parquet format.")

## Load Embeddings Data Frame from Google Drive

In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive')

In [2]:
import pandas as pd

# Ensure pyarrow is installed for Parquet support
# This might already be installed in Colab, but good to ensure
try:
    import pyarrow
except ImportError:
    !pip install pyarrow

embedding_file = '/content/drive/MyDrive/citrus-csv/df_processed_with_embeddings.parquet'
df_with_embeddings = pd.read_parquet(embedding_file)

print(f"Loaded '{embedding_file}' into 'df_with_embeddings'.")
print(f"Shape of df_with_embeddings: {df_with_embeddings.shape}")

Loaded '/content/drive/MyDrive/citrus-csv/df_processed_with_embeddings.parquet' into 'df_with_embeddings'.
Shape of df_with_embeddings: (52108, 26)


In [ ]:
df_with_embeddings.info()

In [ ]:
import os
import pandas as pd

# 1. Define the Output Path
save_dir = '/content/drive/MyDrive/citrus-csv/'
file_name = 'citrus_boolean_control_group.csv'
full_path = os.path.join(save_dir, file_name)

# Ensure directory exists
os.makedirs(save_dir, exist_ok=True)

# 2. Filter the DataFrame
# We select only rows where 'is_boolean_control' is True
df_control = df_with_embeddings[df_with_embeddings['is_boolean_control'] == True].copy()

# 3. Drop Vector Columns for Clean Export
# Removing the 768-dim vector list and the raw embedding text to make the CSV readable
cols_to_drop = ['embedding', 'embedding_text', 'norm_title']
# Only drop if they exist in the df
df_export = df_control.drop(columns=[c for c in cols_to_drop if c in df_control.columns], errors='ignore')

# 4. Save to CSV
df_export.to_csv(full_path, index=False)

# 5. Verification Output
print(f"✅ Export Complete.")
print(f"   Saved {len(df_export)} records to: {full_path}")
print("-" * 30)
print("Preview of Control Group Papers:")
print(df_export[['Year', 'Title', 'DOI']].head())

# Proceed with sqlite-vec

## Setup Sqlite vec

In [ ]:
!pip install sqlite-vec pysqlite3 numpy


In [ ]:
db_path = 'embeddings_db.sqlite'

try:
    # Use pysqlite3 for robust extension loading in Colab
    import pysqlite3 as sqlite3
    print("[INFO] Using pysqlite3 for extension loading.")
except ImportError:
    import sqlite3
    print("[WARNING] pysqlite3 not found. Falling back to standard sqlite3.")

import sqlite_vec
import numpy as np

# Connect to a new (or existing) SQLite database file
conn = sqlite3.connect(db_path)


In [ ]:
# Enable extension loading
conn.enable_load_extension(True)

# Load the sqlite-vec extension
sqlite_vec.load(conn)

# Verify the installation by checking the version
version = conn.execute("SELECT vec_version()").fetchone()[0]
print(f"Using sqlite-vec version: {version}")

# Optionally, disable extension loading after loading for security
conn.enable_load_extension(False)


In [7]:
import sqlite_vec
import sqlite3
import numpy as np

# For persistent storage, use a file path:
db_path = 'embeddings_db.sqlite'

# Connect to a standard sqlite3 database
db = sqlite3.connect(db_path)

# Enable loading extensions (this is necessary for sqlite-vec)
db.enable_load_extension(True)

# Load the sqlite-vec extension into the connection
sqlite_vec.load(db)

# Create a cursor object
cursor = db.cursor()

print(f"Connected to SQLite database: {db_path} and loaded sqlite-vec extension.")

Connected to SQLite database: embeddings_db.sqlite and loaded sqlite-vec extension.


## Create Papers Table

In [ ]:
import pysqlite3 as sqlite3
import sqlite_vec

# Reconnect to the database
db_path = 'embeddings_db.sqlite'
db = sqlite3.connect(db_path)
db.enable_load_extension(True)
sqlite_vec.load(db)
cursor = db.cursor()
print("Reconnected to database and loaded sqlite-vec extension.")

table_name = 'papers'

# Drop table if it exists to ensure a clean start
cursor.execute(f"DROP TABLE IF EXISTS {table_name}")

# Create a table with an 'embedding' vector column with the correct dimension (768)
create_table_sql = f"""
CREATE TABLE {table_name} (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    Authors TEXT,
    Title TEXT,
    Year INTEGER,
    Journal TEXT,
    DOI TEXT,
    Abstract TEXT,
    is_control BOOLEAN,
    embedding VECTOR(768)
);
"""
cursor.execute(create_table_sql)
db.commit()

print(f"Table '{table_name}' dropped and recreated with a VECTOR(768) column for embeddings.")

## Insert Papers with Embeddings

Now that we have our DataFrame with embeddings, we'll save it to a SQLite database using `sqlite-vec`. This allows us to store the textual data along with its corresponding vector embeddings, enabling efficient vector search later.

In [ ]:
import ast # Import the ast module (though we'll use manual parsing for robustness)
import re # Import regex for string manipulation
from tqdm.auto import tqdm # Import tqdm for progress bar

# Prepare the data for insertion
# Ensure embeddings are in a format compatible with sqlite-vec (e.g., numpy array of float32)

insert_count = 0
# Wrap the iteration with tqdm for a progress bar
for index, row in tqdm(df_with_embeddings.iterrows(), total=len(df_with_embeddings), desc="Inserting embeddings"):
    # Directly use the embedding from the DataFrame
    embedding_data = row['embedding']

    # Ensure it's a NumPy array of float32, handling cases where it might be a list or None
    if isinstance(embedding_data, list):
        embedding_np = np.array(embedding_data, dtype=np.float32)
    elif isinstance(embedding_data, np.ndarray):
        embedding_np = embedding_data.astype(np.float32)
    elif embedding_data is None:
        # Handle cases where embedding might be None (e.g., failed embedding generation)
        print(f"Warning: Embedding for row {index} is None. Assigning a zero array.")
        embedding_np = np.zeros(768, dtype=np.float32)
    else:
        print(f"Error: Unexpected embedding type for row {index}: {type(embedding_data)}. Assigning a zero array.")
        embedding_np = np.zeros(768, dtype=np.float32)

    # Pass the numpy array directly to the SQL insert statement
    insert_sql = f"""
    INSERT INTO {table_name} (Authors, Title, Year, Journal, DOI, Abstract, is_control, embedding)
    VALUES (?, ?, ?, ?, ?, ?, ?, ?)
    """
    try:
        cursor.execute(insert_sql, (
            row['Authors'],
            row['Title'],
            row['Year'],
            row['Source title'],
            row['DOI'],
            row['Abstract'],
            row['is_boolean_control'],
            embedding_np # Insert the NumPy array directly
        ))
        insert_count += 1
    except Exception as e:
        print(f"Error inserting row {index}: {e}")

db.commit()
print(f"Inserted {insert_count} records into '{table_name}'.")

# Work with Vectorized Corpus

## [Optional] Query Corpus with Granite Embedding Model

In [ ]:
import pysqlite3 as sqlite3
import numpy as np

from ibm_watsonx_ai import Credentials
from ibm_watsonx_ai.foundation_models import Embeddings
from ibm_watsonx_ai.metanames import EmbedTextParamsMetaNames as EmbedParams
from ibm_watsonx_ai.foundation_models.utils.enums import EmbeddingTypes

from google.colab import userdata
import sqlite_vec

query_string = "Chatbot for human resources"
n_document = 10

WATSONX_URL = userdata.get('WATSONX_URL').split('\r\n')[0].strip()
WATSONX_API_KEY = userdata.get('WATSONX_API_KEY').split('\r\n')[0].strip()
WATSONX_PROJECT_ID = userdata.get('WATSONX_PROJECT_ID').split('\r\n')[0].strip()

def get_watsonx_embedding(text):
    credentials = Credentials(
        url=WATSONX_URL,
        api_key=WATSONX_API_KEY,
    )

    embed_params = {
        EmbedParams.TRUNCATE_INPUT_TOKENS: 512,
        EmbedParams.RETURN_OPTIONS: {
            'input_text': True
        }
    }

    embedding = Embeddings(
        model_id="ibm/slate-125m-english-rtrvr-v2",
        params=embed_params,
        credentials=credentials,
        project_id=WATSONX_PROJECT_ID
    )

    embedding_vector = embedding.embed_query(text=text)

    return embedding_vector

# Generate query embedding using Watsonx.ai
query_embedding_response = get_watsonx_embedding(text=query_string)

# Extract and convert the embedding to a NumPy array
if isinstance(query_embedding_response, list):
    query_embedding = np.array(query_embedding_response, dtype=np.float32)
else:
    raise ValueError("Unexpected response format for query embedding: Expected a list of floats.")

# Reconnect to the database
db = sqlite3.connect(db_path)
db.enable_load_extension(True)
sqlite_vec.load(db)
cursor = db.cursor()

# Store query embedding in a temporary table for proper vector comparison
cursor.execute("DROP TABLE IF EXISTS query_vec_table")
cursor.execute("CREATE TEMPORARY TABLE query_vec_table (query_embedding VECTOR(768))")
cursor.execute("INSERT INTO query_vec_table (query_embedding) VALUES (?) ", (query_embedding,))
db.commit()

# Perform vector similarity search
search_sql = f"""
SELECT
    p.id, p.Authors, p.Title, p.Year, p.Abstract, p.Journal, p.DOI,
    vec_distance_cosine(p.embedding, q.query_embedding) AS distance
FROM {table_name} AS p
JOIN query_vec_table AS q
ORDER BY distance ASC
LIMIT {n_document};
"""

cursor.execute(search_sql)
results = cursor.fetchall()

# Generate markdown output
markdown_output = f"### {query_string.capitalize()} - Top {n_document} Similar Documents\n\n"

for row in results:
    doc_id, authors, title, year, abstract, journal, DOI, distance = row
    markdown_output += f"---\n- **ID**: {doc_id}\n- **Title**: {title}\n- **Authors**: {authors}\n- **Year**: {year}\n- **Journal**: {journal}\n- **DOI**: {DOI}\n- **Abstract**: {abstract}\n- **Distance**: {distance:.4f}\n- **DOI Link**: https://doi.org/{DOI}\n\n"

from IPython.display import display, Markdown
display(Markdown(markdown_output))

# Close the database connection
db.close()

## Methodological Validation: Orthogonality Check

**Objective:** To verify that the defined Semantic Clusters ($Q_1 - Q_5$) represent distinct theoretical territories and do not introduce redundancy.

**The Metric:** Jaccard Similarity Coefficient ($J$)
$$J(A,B) = \frac{|A \cap B|}{|A \cup B|}$$

We retrieve the top-50 documents for each cluster and calculate the pairwise overlap.
*   **Low Overlap ($J < 0.1$):** Indicates distinct sub-fields (Orthogonal).
*   **High Overlap ($J > 0.5$):** Indicates synonymous concepts (Redundant).

*Hypothesis: The "Synthetic Stakeholders" cluster should be distinct from the "Facilitation" cluster.*

In [ ]:
# --- STEP 2: GENERATE FIGURE 1 (ORTHOGONALITY) ---
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np # Ensure numpy is imported
import sqlite3 # Ensure sqlite3 is imported for db connection
import sqlite_vec # Ensure sqlite_vec is imported for extension loading

# The 5 Semantic Clusters defined for the paper
clusters = {
    "Theoretical (Q1)": "Participative decision making frameworks facilitated by generative AI",
    "Synthetic (Q2)": "Using LLM-based agents as synthetic stakeholders in decision making",
    "Facilitation (Q3)": "Conversational AI systems for enhancing employee voice and consensus",
    "Teaming (Q4)": "Human-AI collaboration models for joint problem solving in enterprises",
    "System (Q5)": "Algorithms for aggregating human preferences using natural language processing"
}

def get_top_ids(query_text, k=100):
    """Get top K IDs using IBM Granite embedding + SQLite Search"""
    # 1. Embed query (using your existing function)
    # Note: Ensure get_watsonx_embedding is defined in your session
    q_vec_list = get_watsonx_embedding(query_text)
    q_vec = np.array(q_vec_list, dtype=np.float32) # Convert to NumPy array

    # Reconnect to the database for each call to ensure connection is open if previous cells closed it
    # This is a defensive measure for interactive environments
    db_path = 'embeddings_db.sqlite' # Make sure db_path is defined and correct
    db = sqlite3.connect(db_path)
    db.enable_load_extension(True)
    sqlite_vec.load(db)
    cursor = db.cursor()

    results = db.execute("""
        SELECT id FROM papers
        WHERE vec_distance_cosine(embedding, ?) < 1.0
        ORDER BY vec_distance_cosine(embedding, ?) ASC
        LIMIT ?
    """, [q_vec, q_vec, k]).fetchall()

    db.close() # Close connection after query
    return set([r[0] for r in results])

# Calculate Jaccard
labels = list(clusters.keys())
matrix = np.zeros((len(labels), len(labels)))

print("Calculating Jaccard Overlaps...")
sets = {}
for label, text in clusters.items():
    sets[label] = get_top_ids(text, k=50) # Comparing Top 50

for i, l1 in enumerate(labels):
    for j, l2 in enumerate(labels):
        s1 = sets[l1]
        s2 = sets[l2]
        intersection = len(s1.intersection(s2))
        union = len(s1.union(s2))
        matrix[i, j] = intersection / union if union > 0 else 0

# Plot
plt.figure(figsize=(10, 8))
sns.heatmap(matrix, annot=True, fmt=".2f", cmap="Blues", xticklabels=labels, yticklabels=labels)
plt.title("Figure 1: Cluster Pairwise Retrieval Overlap Analysis (Jaccard)", fontsize=14)
plt.tight_layout()
plt.show()

## The HiLAT Instrument: Expert-Guided Calibration

**Definition:** Human-in-the-Loop Adaptive Truncation (HiLAT).

Standard vector retrieval often relies on arbitrary global thresholds (e.g., $\tau = 0.75$). However, nascent topics often have "steeper" similarity decay curves than established topics.

**The Protocol:**
1.  **Visualization:** The SME (Subject Matter Expert) views the Similarity Decay Curve for a specific cluster.
2.  **Identification:** The SME identifies the "Semantic Cliff"—the inflection point where relevance drops significantly.
3.  **Calibration:** Using the **Coarse-Fine Interface** below, the SME sets the precise retrieval depth ($n$) for that specific cluster.

*This instrument replaces algorithmic guesswork with domain expertise.*

In [ ]:
# --- CITRUS TOOL: PRECISION HiLAT CALIBRATOR ---
import ipywidgets as widgets
from IPython.display import display
import matplotlib.pyplot as plt
import numpy as np

# 1. Ensure Data is Loaded (Defensive check)
if 'decay_curves' not in locals():
    print("Reloading decay curves...")
    # (Re-run the previous data loading block if you lost the session)
    # Assuming decay_curves exists from the previous step to save time

# 2. Create the Widgets
style = {'description_width': 'initial'}

# Dropdown for Cluster
w_cluster = widgets.Dropdown(
    options=list(clusters.keys()),
    description='<b>Select Cluster:</b>',
    style=style,
    layout=widgets.Layout(width='400px')
)

# The Slider (Coarse Control)
w_slider = widgets.IntSlider(
    min=1, max=100, step=1, value=25,
    description='<b>Scan:</b>',
    continuous_update=False, # Wait until release to redraw (faster)
    style=style,
    layout=widgets.Layout(width='500px')
)

# The Stepper (Fine Control - Plus/Minus)
w_stepper = widgets.BoundedIntText(
    min=1, max=100, step=1, value=25,
    description='<b>Precision (n):</b>',
    style=style,
    layout=widgets.Layout(width='200px')
)

# 3. Link Slider and Stepper
# When you move one, the other updates automatically
widgets.link((w_slider, 'value'), (w_stepper, 'value'))

# 4. The Plot Logic
output = widgets.Output()

def update_plot(change=None):
    cluster_name = w_cluster.value
    cutoff_rank = w_slider.value # They are linked, so this works for both

    scores = decay_curves[cluster_name]
    x = range(len(scores))

    # Safety
    if cutoff_rank >= len(scores): cutoff_rank = len(scores) - 1
    selected_score = scores[cutoff_rank]

    with output:
        output.clear_output(wait=True)

        plt.figure(figsize=(10, 5))

        # Plot Curve
        plt.plot(x, scores, color='#2c3e50', linewidth=2, label='Similarity Decay')

        # Plot Cut-off
        plt.axvline(x=cutoff_rank, color='#e74c3c', linewidth=2, label=f'Cut-off (n={cutoff_rank})')
        plt.axhline(y=selected_score, color='#27ae60', linestyle='--', label=f'Threshold ({selected_score:.3f})')

        # Plot "Keep Zone" vs "Drop Zone" shading
        plt.axvspan(0, cutoff_rank, color='#27ae60', alpha=0.1, label='Keep Zone')
        plt.axvspan(cutoff_rank, 100, color='#e74c3c', alpha=0.05, label='Drop Zone')

        plt.title(f"Researcher Calibrated: {cluster_name}", fontsize=14, fontweight='bold')
        plt.xlabel("Document Rank", fontsize=10)
        plt.ylabel("Cosine Similarity", fontsize=10)
        plt.xlim(0, 100)
        plt.legend(loc='upper right')
        plt.grid(True, alpha=0.3)
        plt.show()

        # Print Stats Table
        print(f"CALIBRATION REPORT FOR '{cluster_name}'")
        print("-" * 40)
        print(f"  • Selected Rank (n):      {cutoff_rank}")
        print(f"  • Similarity Threshold:   {selected_score:.4f}")
        print("-" * 40)

# 5. Connect Events
w_cluster.observe(update_plot, names='value')
w_slider.observe(update_plot, names='value')

# 6. Display Layout
ui = widgets.VBox([
    w_cluster,
    widgets.HBox([w_slider, w_stepper]), # Put Slider and Stepper side-by-side
    output
])

# Initialize
update_plot()
display(ui)

## Phase III: Calibrated Acquisition & Reporting

**Objective:** To generate the final Bibliometric SOTA (State of the Art) list.

Using the cut-off parameters ($n$) determined in the previous HiLAT step, we perform the final retrieval from the vector database.

**The "Hydration" Process:**
1.  **Retrieve:** Fetch the top $n$ IDs for each cluster.
2.  **Hydrate:** Join these IDs with the original metadata (Authors, Journal, Year) from the raw DataFrame.
3.  **Deduplicate:** Ensure no single paper is counted twice if it appears in multiple clusters.
4.  **Format:** Generate APA-style citations for immediate insertion into the manuscript Appendix.

In [ ]:
# --- CITRUS STEP 5: FINAL ACQUISITION (BASED ON RESEARCHER CALIBRATION) ---
import pandas as pd
import numpy as np
import sqlite3
import sqlite_vec

# 1. INPUT: Your Calibrated Cut-offs (From the Graphs)
final_calibration = {
    "Theoretical (Q1)": 10,  # Researcher enter value based on human calibration
    "Synthetic (Q2)":   20,  # Researcher enter value based on human calibration
    "Facilitation (Q3)": 11, # Researcher enter value based on human calibration
    "Teaming (Q4)":     9,   # Researcher enter value based on human calibration
    "System (Q5)":      8    # Researcher enter value based on human calibration
}

# 2. SETUP: Connect to DB
db_path = 'embeddings_db.sqlite'
db = sqlite3.connect(db_path)
db.enable_load_extension(True)
sqlite_vec.load(db)
cursor = db.cursor()

acquired_papers = []

# --- CITRUS TOOL: APA CITATION GENERATOR ---
def generate_apa(row):
    """
    Constructs a rough APA citation from metadata.
    Format: Author(s). (Year). Title. Journal/Source. DOI.
    """
    # 1. Process Authors (Scopus often gives "Smith, J.; Doe, J.")
    # We'll take the first author et al. for brevity if >3, or format nicely
    authors = row.get('Authors', 'Unknown Authors')
    if pd.isna(authors): authors = "Unknown"

    author_list = str(authors).split(';')
    if len(author_list) > 1:
        # Take first author and add et al.
        first_author = author_list[0].strip()
        citation_author = f"{first_author} et al."
    else:
        citation_author = author_list[0].strip()

    # 2. Process Year
    year = str(row.get('Year', 'n.d.'))

    return f"{citation_author} ({year})"


print("--- STARTING RETRIEVAL ---")

# 3. EXECUTION: Retrieve specific N for each cluster
for label, n_cutoff in final_calibration.items():
    if n_cutoff == 0: continue

    # Get query vector
    query_text = clusters[label]
    q_vec_list = get_watsonx_embedding(query_text)
    q_vec = np.array(q_vec_list, dtype=np.float32)

    # Execute Vector Search LIMIT n_cutoff
    results = db.execute("""
        SELECT
            doi,
            title,
            authors,
            year,
            abstract,
            vec_distance_cosine(embedding, ?) as dist
        FROM papers
        ORDER BY dist ASC
        LIMIT ?
    """, [q_vec, n_cutoff]).fetchall()

    print(f"Cluster '{label}': Retrieved Top {n_cutoff} docs.")

    # Store results
    for r in results:
        doi, title, authors, year, abstract, dist = r
        sim = 1 - dist

        acquired_papers.append({
            "Cluster_Source": label,
            "DOI": doi,
            "Title": title,
            "Authors": authors,
            "Year": year,
            "Similarity": sim,
            "Abstract": abstract
        })

db.close()

# 4. PROCESSING: Deduplicate (in case a paper appeared in two clusters)
df_final = pd.DataFrame(acquired_papers)

# Check for duplicates (e.g., if a paper was in top 10 of Q1 AND top 20 of Q2)
initial_len = len(df_final)
df_final = df_final.sort_values(by='Similarity', ascending=False)
df_final = df_final.drop_duplicates(subset=['DOI'], keep='first')

# Apply to citation formating
df_final['APA_Citation'] = df_final.apply(generate_apa, axis=1)

final_len = len(df_final)

# 5. EXPORT
csv_filename = "citrus_final_screening_list.csv"
df_final.to_csv(csv_filename, index=False)

print("\n" + "="*40)
print(f"Total Retrieved Candidates: {initial_len}")
print(f"Unique Documents (Deduplicated): {final_len}")
print(f"Results saved to: {csv_filename}")
print("="*40)

# Display Preview
df_final[['Similarity', 'Cluster_Source', 'Title', 'Authors', 'Year']].head(10)

## Artifact Generation

**Outputs:**
1.  `citrus_final_screening_list.csv`: The raw dataset for full-text review.
2.  **Recall Statistics:** Comparison of the Retrieved Set against the Boolean Control Group (to calculate Recall Gain).

*Next Step: The researcher manually reviews the Abstract/Full-Text of these candidates to populate the PRISMA Inclusion node.*

In [ ]:
import shutil
import os

source_file = 'citrus_final_screening_list.csv'
destination_path = file_path # This variable holds '/content/drive/MyDrive/citrus-csv'

# Ensure the destination directory exists (though file_path usually exists from drive.mount)
os.makedirs(destination_path, exist_ok=True)

# Construct the full destination file path
destination_file = os.path.join(destination_path, source_file)

# Move the file
shutil.move(source_file, destination_file)

print(f"'{source_file}' successfully moved to '{destination_file}'")

In [ ]:
# @title 📦 Step 6: Download the "Glass Box" (Epistemic Artifact)

import os
import shutil
from google.colab import files

# 1. Identify the Database File
# (Defaults to the name used in Step 2)
target_db = 'embeddings_db.sqlite' # Corrected to the actual database file name

if os.path.exists(target_db):
    print(f"📦 Packaging the Digital Artifact: {target_db}... (Note: This database contains all columns, including 'Abstract'.)")

    # 2. Compress the file (GZIP)
    # SQLite files are compressible (~50% reduction), saving bandwidth
    artifact_name = "citrus_epistemic_artifact.sqlite.gz"
    !gzip -c "{target_db}" > "{artifact_name}"

    # 3. Calculate Stats
    original_size = os.path.getsize(target_db) / (1024 * 1024)
    compressed_size = os.path.getsize(artifact_name) / (1024 * 1024)

    print(f"   Original Size:   {original_size:.2f} MB")
    print(f"   Compressed Size: {compressed_size:.2f} MB")

    # 4. Trigger Download
    print(f"⬇️ Downloading {artifact_name}...")
    files.download(artifact_name)

else:
    print(f"❌ Error: Database file '{target_db}' not found. Did you run Step 2?")


In [ ]:
# @title 📦 Step 6: Download the "Sanitized" Glass Box (Epistemic Artifact)

import os
import shutil
import sqlite3
import pandas as pd
from google.colab import files

# 1. Identify the Source Database
source_db = 'embeddings_db.sqlite' # Corrected to the actual database file name
sanitized_db = "citrus_public_artifact.sqlite"

if os.path.exists(source_db):
    print(f"🛡️  Initiating Sanitization Protocol on {source_db}...")

    # 2. Connect and Create Sanitized Copy
    # We use SQL to copy everything EXCEPT the 'Abstract' column
    conn_src = sqlite3.connect(source_db)
    cursor_src = conn_src.cursor()

    # Get all column names
    cursor_src.execute("PRAGMA table_info(papers)")
    columns = [info[1] for info in cursor_src.fetchall()]

    # Filter out 'abstract' (case insensitive)
    # This keeps Vectors, DOIs, Titles, Clusters, etc.
    keep_cols = [c for c in columns if c.lower() != 'abstract']
    cols_sql = ", ".join(keep_cols)

    print(f"   ℹ️  Retaining columns: {keep_cols}")
    print(f"   ✂️  Redacting column: ['abstract'] for Copyright Compliance.")

    # Create new DB and copy data
    if os.path.exists(sanitized_db):
        os.remove(sanitized_db)

    conn_dest = sqlite3.connect(sanitized_db)

    # Attach destination to source for fast transfer
    conn_src.execute(f"ATTACH DATABASE '{sanitized_db}' AS dest_db")

    # Copy structure and data (minus abstract)
    conn_src.execute(f"CREATE TABLE dest_db.papers AS SELECT {cols_sql} FROM main.papers")

    # Commit and detach
    conn_src.commit()
    conn_src.execute("DETACH DATABASE dest_db")
    conn_src.close()
    conn_dest.close()

    print("   ✅ Sanitization Complete.")

    # 3. Compress the file (GZIP)
    artifact_name = "citrus_epistemic_artifact.sqlite.gz"
    print(f"📦 Compressing artifact...")
    !gzip -c "{sanitized_db}" > "{artifact_name}"

    # 4. Calculate Stats & Savings
    original_size = os.path.getsize(source_db) / (1024 * 1024)
    sanitized_size = os.path.getsize(sanitized_db) / (1024 * 1024)
    compressed_size = os.path.getsize(artifact_name) / (1024 * 1024)

    print(f"\n📊 Artifact Statistics:")
    print(f"   Original Size (Private):   {original_size:.2f} MB")
    print(f"   Sanitized Size (Public):   {sanitized_size:.2f} MB (Abstracts Removed)")
    print(f"   Final Download Size:       {compressed_size:.2f} MB")

    # 5. Trigger Download
    print(f"⬇️ Downloading {artifact_name}...")
    files.download(artifact_name)

else:
    print(f"❌ Error: Source database '{source_db}' not found. Did you run the ingestion/embedding steps?")